# 🧠 Notebook 04 — RFM + Cohort Prep (Pandas)
Creates two outputs used by Dashboard 2:
- `dw_customer_rfm.csv`
- `cohort_retention_matrix.csv`

**Concepts called out:**
- `merge` for joining tables
- Derived columns (revenue)
- `groupby + agg` including custom `lambda`
- Bucketing with `pd.qcut`
- Cohort logic using period arithmetic
- Pivot table creation with `.pivot`


In [ ]:
import pandas as pd
from pathlib import Path


## 1) Load raw data

In [ ]:
RAW = Path('../data/raw')
orders = pd.read_csv(RAW/'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(RAW/'order_items.csv')
orders.head()

## 2) Merge and compute revenue (transaction-level)
**Business context:** Monetary value comes from sum of item revenues.

In [ ]:
df = orders.merge(order_items, on='order_id', how='inner')
df['revenue'] = df['quantity'] * df['item_price']
df = df[df['order_status'].isin(['paid','shipped','delivered'])].copy()
df.head()

## 3) RFM as of a snapshot date
Change snapshot_date to test different scoring dates.

In [ ]:
snapshot_date = pd.Timestamp('2024-12-31')
rfm = df.groupby('customer_id').agg(
    recency_days=('order_date', lambda s: (snapshot_date - s.max()).days),
    frequency_orders=('order_id','nunique'),
    monetary_spend=('revenue','sum')
).reset_index()
rfm.head()

## 4) Bucket + Score
**Concepts:** quartiles; higher recency rank = more recent buyer.

In [ ]:
rfm['r_rank'] = pd.qcut(rfm['recency_days'], 4, labels=[4,3,2,1]).astype(int)
rfm['f_rank'] = pd.qcut(rfm['frequency_orders'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['m_rank'] = pd.qcut(rfm['monetary_spend'], 4, labels=[1,2,3,4]).astype(int)
rfm['rfm_score'] = rfm['r_rank']*100 + rfm['f_rank']*10 + rfm['m_rank']
rfm.sort_values('rfm_score', ascending=False).head(10)

## 5) Cohort retention matrix
**Business context:** Cohorts show if newer customers are retained as well as older cohorts.

In [ ]:
co = orders[orders['order_status'].isin(['paid','shipped','delivered'])].copy()
co['order_month'] = co['order_date'].dt.to_period('M')
co['cohort_month'] = co.groupby('customer_id')['order_month'].transform('min')
co['months_since'] = (co['order_month'] - co['cohort_month']).apply(lambda p: p.n)

cohort_counts = co.groupby(['cohort_month','months_since']).agg(customers=('customer_id','nunique')).reset_index()
cohort_sizes = cohort_counts[cohort_counts['months_since']==0][['cohort_month','customers']].rename(columns={'customers':'cohort_size'})
cohort_counts = cohort_counts.merge(cohort_sizes, on='cohort_month', how='left')
cohort_counts['retention_pct'] = cohort_counts['customers'] / cohort_counts['cohort_size']

matrix = cohort_counts.pivot(index='cohort_month', columns='months_since', values='retention_pct')
matrix.head()

## 6) Export outputs for Power BI

In [ ]:
OUT = Path('../data/processed')
OUT.mkdir(parents=True, exist_ok=True)
rfm.to_csv(OUT/'dw_customer_rfm.csv', index=False)
matrix.reset_index().to_csv(OUT/'cohort_retention_matrix.csv', index=False)
print('Saved:', OUT/'dw_customer_rfm.csv')
print('Saved:', OUT/'cohort_retention_matrix.csv')